In [1]:
import warnings
warnings.filterwarnings('ignore')

(Cookbook_Group_Free_Hierarchy)=
# Preserving molecules when adding groups

*Staging group hierarchy and using external selectors without changing molecular identity.*

Converted ligand graphs can have valid molecule membership with no groups or
chains. You can add groups through the native builder while retaining source
molecule indices and labels. MDTraj selections also work before groups are
added, using target-only placeholders rather than changing your native system.

:::{versionadded} 1.0.0
:::


## Starting with complete molecular graphs

The input has two connected molecules, a declared fixed chemical state and no
native groups. No coordinates or molecular preparation are needed.


In [2]:
import molsysmt as msm
from rdkit import Chem

molsys = msm.convert(Chem.MolFromSmiles('CCO.COC'), to_form='molsysmt.MolSys')
source_ids = molsys.topology.molecules.molecule_id.tolist()
print(msm.get(molsys, n_molecules=True, n_groups=True))


[2, 0]


## Selecting through MDTraj

MDTraj requires a residue and chain for every target atom. For missing native
groups, the adapter creates UNL placeholders per known molecule and chain, or
per atom when molecule membership is unknown. Missing chains use a fallback
chain. These parents exist only in the external target. UNL does not imply a
protein or water assignment. Selectors such as `resid` use this target metadata,
not missing native group indices. A declared group with conflicting known
chains is rejected rather than silently reassigned.


In [3]:
selected = msm.select(molsys, selection='index 0', syntax='MDTraj')
external = msm.convert(molsys, to_form='mdtraj.Topology')
print(selected)
print([[atom.index for atom in group.atoms] for group in external.residues])
print(msm.select(molsys, selection='protein', syntax='MDTraj'))
print(msm.get(molsys, n_groups=True, n_chains=True))


[0]
[[0, 1, 2], [3, 4, 5]]
[]
[0, 0]


## Adding groups to existing molecules

A source group-free molecule retains its identity when its atoms are assigned
to groups. The group names below are explicit illustrative labels. The builder
does not add duplicate molecule rows or reinterpret reference membership using
another chemical state's components. Source IDs remain string labels.


In [4]:
builder = msm.build.editable(molsys)
builder.add_group([0, 1, 2], group_name='LIG', group_id='ethanol')
builder.add_group([3, 4, 5], group_name='LIG', group_id='ether')
new_molsys = builder.build()
assert new_molsys.topology.molecules.molecule_id.tolist() == source_ids
print(msm.get(new_molsys, n_molecules=True, n_groups=True))
print(msm.get(new_molsys, element='molecule', atom_index=True))


[2, 2]
[[0, 1, 2], [3, 4, 5]]


## Completing a partial grouping

Building creates the ordinary fallback groups for remaining ungrouped source
atoms, while retaining their original molecule ownership. Repeated builds and
independent builder copies retain the same molecule axis.


In [5]:
partial_builder = msm.build.editable(molsys)
partial_builder.add_group([2], group_name='LIG')
partial = partial_builder.build()
print(msm.get(partial, n_molecules=True))
print(msm.get(partial, element='molecule', atom_index=True))


2
[[0, 1, 2], [3, 4, 5]]


## Rejecting conflicting groups

A group cannot span different inherited source molecules. Rejection occurs
before the group or atom memberships are changed. Explicit new systems assembled
without inherited molecule membership retain their ordinary builder inference.


In [6]:
builder = msm.build.editable(molsys)
try:
    builder.add_group([0, 3], group_name='LIG')
except msm.StructuralInconsistencyError:
    print('Cross-molecule grouping rejected; groups:', builder.n_groups)


Cross-molecule grouping rejected; groups: 0


## Calculating on the preserved molecules

Both the external selector and the compiled group hierarchy preserve access to
the complete source molecules. TPSA remains an area-valued graph descriptor;
the choice of selection syntax does not change its chemical model.


In [7]:
selected_area = msm.physchem.get_tpsa(
    molsys, selection='index 0', syntax='MDTraj',
)
areas = msm.physchem.get_tpsa(new_molsys)
print(msm.pyunitwizard.get_value(selected_area, to_unit='angstrom**2').round(2).tolist())
print(msm.pyunitwizard.get_value(areas, to_unit='angstrom**2').round(2).tolist())


[20.23]
[20.23, 9.23]


## Appending atoms to populated chemistry

You can extend the atom inventory without losing existing state assignments.
All states gain nullable rows before the operation is published. The new atom
still needs explicit chemistry; element type alone does not determine formal
charge, hydrogen count or chemical preparation. Coordinates, velocities and
B factors prohibit this atom-axis change instead of being resized implicitly.

A group-free source can also remain group-free after compilation. New disconnected
components receive extra molecule rows while inherited molecule indices and
labels are retained, including for explicitly permuted source atom inventories.
Connectivity edits that split or merge inherited group-free molecules require
explicit groups to retain their boundaries rather than silently changing the axis.


In [8]:
sparse_builder = msm.build.editable(molsys)
sparse_builder.add_atom(atom_name='Ar', atom_type='Ar')
sparse_molsys = sparse_builder.build()
print(msm.get(sparse_molsys, n_atoms=True, n_groups=True, n_molecules=True))
print(msm.get(sparse_molsys, element='molecule', atom_index=True))


[7, 0, 3]
[[0, 1, 2], [3, 4, 5], [6]]


In [9]:
builder = msm.build.editable(molsys)
builder.add_group([0, 1, 2], group_name='LIG')
builder.add_group([3, 4, 5], group_name='LIG')
new_atom = builder.add_atom(atom_id='argon', atom_name='Ar', atom_type='Ar')
print(msm.get(builder, element='atom', selection=[new_atom], formal_charge=True))
new_group = builder.add_group([new_atom], group_name='AR')
builder.add_molecule([new_group], molecule_id='argon', molecule_name='argon')
new_molsys = builder.build()
print(msm.get(new_molsys, n_atoms=True, n_molecules=True))
print(msm.get(new_molsys, element='molecule', atom_index=True))
print(msm.get(molsys, n_atoms=True))


[None]
[7, 3]
[[0, 1, 2], [3, 4, 5], [6]]
6


Grouping and atom-axis growth are separate operations. Atom growth now retains
all existing states but leaves new chemistry unknown; it is not molecular
preparation. Scientific readers can reject the resulting partial chemistry until
it is explicitly completed and falls within their supported method profile.
Native subset
extraction retains its existing order policy; use {func}`molsysmt.basic.permute_atoms`
when you explicitly need to reorder the complete atom axis before conversion.

:::{seealso}
:class: dropdown

- {ref}`Tutorial_Select` for native and external input syntaxes.
- {ref}`Tutorial_Editable` for staging and compiling systems.
- {ref}`Tutorial_Get_tpsa` for descriptor chemistry and units.
- {ref}`user-foundations-support-selection-syntaxes` for target metadata limits.
:::


## Declaring ownership without groups

You can declare molecule membership even when the graph is incomplete or no
bonds are known. The destination molecules must already exist. Indices are
positions, not molecule IDs. `None` keeps an atom's ownership unknown. Changing
a chemical component afterwards does not overwrite this stable declaration.
Use `atom_indices` for a partial update, or pass `None` to clear selected
ungrouped assignments. A grouped position must agree with its group's owner.

{ref}`API Native Molecule Membership` describes validation and analysis
invalidation. H5MSM 0.5 preserves independent declarations in its topology layer;
legacy dictionary schema 0.1 rejects declarations it cannot represent.


In [10]:
from molsysmt.native import MolSys

molsys_A = MolSys(n_atoms=3, n_molecules=2)
molsys_A.topology.molecules['molecule_id'] = ['ligand', 'ion']
molsys_A.topology.set_atom_molecules([0, 0, None])
molsys_A.chemical_states.add_bonds([[0, 1]])
print(molsys_A.topology.get(element='atom', molecule_index=True))
molsys_A.chemical_states.remove_bonds([0])
print(molsys_A.topology.get(element='atom', molecule_index=True))
molsys_A.topology.set_atom_molecules(1, atom_indices=2)
print(molsys_A.topology.get(element='atom', molecule_id=True))
molsys_A.topology.set_atom_molecules(None, atom_indices=[2])
print(molsys_A.topology.get(element='atom', molecule_index=True))

[0, 0, None]
[0, 0, None]
['ligand', 'ligand', 'ion']
[0, 0, None]


## Restricting complete and partial graphs

The original graph-backed input has complete connectivity for its declared atom
domain. Selecting atoms 0 and 2 removes the atom between them, splitting their
component while retaining one stable molecule. Complete induced connectivity
does not certify a prepared standalone fragment: terminal chemistry is not
repaired. Names/types of changed components remain unknown.

For comparison, explicitly declaring partial connectivity makes the same
selection restrict the original component membership. The cut declaration is
partial, not a newly inferred complete partition. Source state order, reference
and structure associations are preserved. See {ref}`Tutorial_Extract` and the
{ref}`MolSys foundation <user-foundations-native-world-classes-molsysmt-molsys>`.


In [11]:
molsys_B = msm.extract(molsys, selection=[0, 2])
print(msm.get(molsys_B, n_components=True, n_molecules=True))
print(msm.get(molsys_B, element='atom', molecule_index=True))

state_records = msm.convert(molsys.chemical_states, to_form='molsysmt.ChemicalStatesDict')
state_records.data['states'][0]['connectivity_completeness'] = 'partial'
molsys_C = msm.copy(molsys)
molsys_C.chemical_states = msm.convert(state_records, to_form='molsysmt.ChemicalStates')
molsys_C = msm.extract(molsys_C, selection=[0, 2])
print(msm.get(molsys_C, n_components=True, n_molecules=True))
print(msm.get(molsys_C, component_completeness=True))
print(msm.get(molsys, n_components=True, n_molecules=True))

[2, 1]
[0, 0]


[1, 1]
partial
[2, 2]


## Editing a selected chemical state

An explicitly appended state starts with unknown chemistry, not a copy of another
state. Here you declare a few edges in that alternative and then remove one of
them by its state-local bond index. The reference state, stable molecules and
original source remain unchanged. These are stored-graph edits, not chemical
state enumeration or preparation. With no complete edge evidence, rebuilt
components remain partial.

For an attached analysis, graph/completeness changes invalidate its evaluated
scope, including structures with zero occurrences. Previously held analysis
objects remain snapshots; coordinates are not copied. Label-only reconstruction
does not invalidate observations. Validation, reconstruction or invalidation
allocation failure preserves the preceding chemistry and analyses. See
{ref}`Tutorial_Native_Chemical_Editing` and {ref}`API Native Chemical Editing`.
Use the controlled methods rather than modifying the live `get_bonds()` table.


In [12]:
molsys_B = msm.copy(molsys)
alternative = molsys_B.chemical_states.append_state()
molsys_B.chemical_states.add_bonds([[0, 1], [1, 2]], chemical_state=alternative)
print([len(molsys_B.chemical_states.get_bonds(chemical_state=i)) for i in (0, alternative)])
molsys_B.chemical_states.remove_bonds([0], chemical_state=alternative)
molsys_B.chemical_states.rebuild_components(
    False, True, False, False, chemical_state=alternative,
)
print([len(molsys_B.chemical_states.get_bonds(chemical_state=i)) for i in (0, alternative)])
print(molsys_B.chemical_states.reference_chemical_state_index)
print(msm.get(molsys_B, element='atom', molecule_index=True))
print(molsys.chemical_states.n_chemical_states)

[4, 2]
[4, 1]
0
[0, 0, 0, 1, 1, 1]
1


## Inspecting an alternate state

The alternate graph now has its own bond/component indices. Inspect them directly
on its chemical collection, while the stable molecule declarations above remain
unchanged. Incident counts include edges to unselected atoms; the reference stays
fixed. The graph is still partial, not a certificate of chemical preparation.


In [13]:
chemical_states = molsys_B.chemical_states
print(chemical_states.get(chemical_state=alternative, n_bonds=True,
    n_components=True, connectivity_completeness=True))
print(chemical_states.get(element='atom', selection=[2, 0, 2],
    chemical_state=alternative, component_index=True, n_bonds=True))
assert msm.get(chemical_states, chemical_state=alternative, n_bonds=True) == 1
assert chemical_states.reference_chemical_state_index == 0


[1, 5, 'partial']
[[1, 0, 1], [1, 0, 1]]


## Inspecting stored assignments

The imported reference state carries RDKit's declared integer formal charges
(in elementary-charge units) and aromaticity. Its known zero charges and False
flags differ from the unassigned alternative state. Repeated atom positions
remain repeated in query output. Unknown bond order is not inferred as a single
bond, and reading these values does not recalculate the graph or invoke RDKit.


In [14]:
print(chemical_states.get(element='atom', selection=[2, 0, 2],
    chemical_state=0, formal_charge=True, atom_is_aromatic=True))
print(chemical_states.get(element='atom', selection=[2, 0, 2],
    chemical_state=alternative, formal_charge=True, atom_is_aromatic=True))
print(chemical_states.get(element='bond', chemical_state=alternative,
    bond_order=True, bond_stereo_atom_indices=True))
assert chemical_states.get(chemical_state=0, formal_charge=True) == [0] * 6
assert chemical_states.reference_chemical_state_index == 0


[[0, 0, 0], [False, False, False]]
[None, None]
[[None], [[None, None]]]


## Assigning the alternative state

Assign selected atom and bond properties through the collection without changing
the imported reference state. Formal charges remain integer multiples of the
elementary charge. Unselected alternative-state atoms stay unknown. These
assignments declare values; they do not add hydrogens, perceive aromaticity or
prepare the molecule. Coordinate arrays are not involved. Rebuild components
explicitly if you change their bond-participation criterion.


In [15]:
msm.set(chemical_states, selection=[2, 0], chemical_state=alternative,
    formal_charge=[-1, 1])
msm.set(chemical_states, element='bond', chemical_state=alternative, bond_order=2)
print(chemical_states.get(chemical_state=alternative, formal_charge=True))
print(chemical_states.get(element='bond', chemical_state=alternative, bond_order=True))
assert chemical_states.get(chemical_state=0, formal_charge=True) == [0] * 6
assert chemical_states.reference_chemical_state_index == 0


[1, None, -1, None, None, None]
[2]


## Declaring partial components

A state-local component row can carry a label through selected member atoms.
All of that component's members read the same label. Clearing one atom's
membership changes its old full atom set and removes the corresponding metadata;
the untouched singleton component retains its label. Inventory size stays fixed,
including empty rows. Stable molecule ownership and bonds remain unchanged.
The declaration is partial/user_defined; graph reconstruction is explicit.


In [16]:
msm.set(chemical_states, element='atom', selection=[0],
    chemical_state=alternative, component_name='untouched singleton')
msm.set(chemical_states, element='atom', selection=[1],
    chemical_state=alternative, component_name='old pair')
msm.set(chemical_states, selection=[1], chemical_state=alternative, component_index=None)
print(chemical_states.get(element='atom', chemical_state=alternative, component_index=True))
print(chemical_states.get(element='atom', selection=[0, 1, 2],
    chemical_state=alternative, component_name=True))
assert chemical_states.get(chemical_state=alternative, n_components=True) == 5
assert chemical_states.get(chemical_state=alternative, component_evidence=True) == 'user_defined'
assert chemical_states.reference_chemical_state_index == 0


[0, None, 1, 2, 3, 4]
['untouched singleton', None, None]


## Reading the typed transport payload

Encode the declared states once, then inspect only the requested typed columns.
The alternate state's partial memberships and nullable assignments survive.
Reading does not reconstruct graphs, prepare the molecule or change the reference.
Use `msm.set` for controlled field assignments; convert to `ChemicalStates`
for graph-editing methods.

In [17]:
payload = msm.convert(chemical_states, to_form='molsysmt.ChemicalStatesDict')
print(payload.get(element='atom', selection=[2, 0, 2],
    chemical_state=alternative, formal_charge=True, component_index=True))
print(payload.get(chemical_state=alternative,
    n_bonds=True, n_components=True, component_evidence=True))
assert payload.get(element='atom', selection=[2, 0, 2],
    chemical_state=alternative, formal_charge=True) == [-1, 1, -1]
assert payload.get(reference_chemical_state_index=True) == 0
assert chemical_states.reference_chemical_state_index == 0

[[-1, 1, -1], [1, 0, 1]]
[1, 5, 'user_defined']


:::{seealso}
:class: dropdown
**Related tools and references**

- {func}`molsysmt.basic.get` and {meth}`molsysmt.native.ChemicalStates.get` query stored counts and membership.
- {meth}`molsysmt.native.ChemicalStates.add_bonds`, {meth}`molsysmt.native.ChemicalStates.remove_bonds` and {meth}`molsysmt.native.ChemicalStates.rebuild_components` edit a selected graph.
- {ref}`Tutorial_Native_Chemical_Editing` explains controlled publication and completeness.
:::
